# Phase and transition analysis

This notebook reproduces the phase and transition analyses reported in the
paper. Phase-change comparisons use the axon as the statistical unit. The
transition analysis pools transition events within each condition and applies
the same two-proportion tests used for the paper.

Required input columns:

- `Condition`: experimental condition;
- `t0`, `t1`, ..., `tN`: phase at each time point, encoded as `-1`, `0`, or `1`.

This notebook is designed for the included
`data/example_synthetic_phase_data.xlsx` dataset. Results are written to
`results/phase_transition_results.xlsx`.


## Configuration and reusable functions


In [1]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import kruskal, mannwhitneyu, norm


REPOSITORY_ROOT = Path.cwd()
if not (REPOSITORY_ROOT / "data").exists() and (REPOSITORY_ROOT.parent / "data").exists():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent

DATA_FILE = REPOSITORY_ROOT / "data" / "example_synthetic_phase_data.xlsx"
OUTPUT_DIR = REPOSITORY_ROOT / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONDITION_COLUMN = "Condition"
EXPECTED_CONDITIONS = [
    "CONTROL",
    "TREATMENT_W",
    "TREATMENT_X",
    "TREATMENT_Y",
    "TREATMENT_Z",
]
TIME_COLUMNS = [f"t{index}" for index in range(29)]


def adjust_pvalues_bh(pvalues):
    """Benjamini-Hochberg false-discovery-rate correction."""
    pvalues = np.asarray(pvalues, dtype=float)
    order = np.argsort(pvalues)
    ranked = pvalues[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    output = np.empty_like(adjusted)
    output[order] = np.minimum(adjusted, 1.0)
    return output


def pairwise_mann_whitney(data, value_column, group_column="Condition"):
    """Run all pairwise Mann-Whitney tests and apply BH-FDR correction."""
    rows = []
    groups = list(dict.fromkeys(data[group_column]))
    for first, second in combinations(groups, 2):
        first_values = data.loc[data[group_column] == first, value_column].dropna()
        second_values = data.loc[data[group_column] == second, value_column].dropna()
        statistic, pvalue = mannwhitneyu(first_values, second_values, alternative="two-sided")
        rows.append({
            "Condition 1": first, "Condition 2": second,
            "n1": len(first_values), "n2": len(second_values),
            "Median 1": first_values.median(), "Median 2": second_values.median(),
            "Mann-Whitney U": statistic, "p": pvalue,
        })
    results = pd.DataFrame(rows)
    results["q (BH-FDR)"] = adjust_pvalues_bh(results["p"])
    results["Significant after FDR"] = results["q (BH-FDR)"] < 0.05
    return results.sort_values("p").reset_index(drop=True)


def compare_two_proportions(x1, n1, x2, n2):
    """Two-sided pooled two-proportion z-test and 95% CI for the difference."""
    p1, p2 = x1 / n1, x2 / n2
    pooled = (x1 + x2) / (n1 + n2)
    pooled_se = np.sqrt(pooled * (1 - pooled) * (1 / n1 + 1 / n2))
    z = (p1 - p2) / pooled_se if pooled_se > 0 else 0.0
    pvalue = 2 * norm.sf(abs(z))
    difference_se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    difference = p1 - p2
    return p1, p2, difference, z, pvalue, difference - 1.96 * difference_se, difference + 1.96 * difference_se


## Load and validate the data


In [3]:
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Example dataset not found: {DATA_FILE}")

data = pd.read_excel(DATA_FILE)
if CONDITION_COLUMN not in data.columns:
    raise ValueError(f"Missing required column: {CONDITION_COLUMN}")

missing_time_columns = [column for column in TIME_COLUMNS if column not in data.columns]
if missing_time_columns:
    raise ValueError(f"Missing time columns: {missing_time_columns}")

data[CONDITION_COLUMN] = data[CONDITION_COLUMN].astype(str).str.strip().str.upper()
observed_conditions = list(pd.unique(data[CONDITION_COLUMN]))
if set(observed_conditions) != set(EXPECTED_CONDITIONS):
    raise ValueError(
        f"Expected conditions {EXPECTED_CONDITIONS}, found {observed_conditions}"
    )

conditions = EXPECTED_CONDITIONS
data[CONDITION_COLUMN] = pd.Categorical(data[CONDITION_COLUMN], categories=conditions, ordered=True)

phases = data[TIME_COLUMNS].apply(pd.to_numeric, errors="coerce").to_numpy()
if np.isnan(phases).any():
    raise ValueError("Phase columns contain missing or non-numeric values.")

observed_states = set(np.unique(phases).astype(int))
if not observed_states.issubset({-1, 0, 1}):
    raise ValueError(f"Unexpected phase states: {sorted(observed_states)}")

print(f"Loaded {len(data)} axons, {len(TIME_COLUMNS)} time points, and {len(conditions)} conditions.")
print(data[CONDITION_COLUMN].value_counts(sort=False))


Loaded 285 axons, 29 time points, and 5 conditions.
Condition
CONTROL        64
TREATMENT_W    63
TREATMENT_X    60
TREATMENT_Y    51
TREATMENT_Z    47
Name: count, dtype: int64


## Phase changes and transitions


In [4]:
# Phase changes per axon
data["Phase changes"] = np.sum(np.diff(phases, axis=1) != 0, axis=1)
phase_summary = (
    data.groupby(CONDITION_COLUMN, observed=True)["Phase changes"]
    .agg(n="size", mean="mean", sd="std", median="median", minimum="min", maximum="max")
    .reset_index()
)

phase_groups = [data.loc[data[CONDITION_COLUMN] == condition, "Phase changes"] for condition in conditions]
kw_statistic, kw_pvalue = kruskal(*phase_groups)
phase_global = pd.DataFrame([{"Test": "Kruskal-Wallis", "Statistic": kw_statistic, "p": kw_pvalue}])
phase_pairwise = pairwise_mann_whitney(data, "Phase changes")


# Pooled transition counts by condition
states = (-1, 0, 1)
transition_names = [f"{start}->{end}" for start in states for end in states]
transition_counts = pd.DataFrame(0, index=conditions, columns=transition_names, dtype=int)
for condition in conditions:
    condition_phases = phases[np.asarray(data[CONDITION_COLUMN] == condition)]
    for axon in condition_phases:
        for start, end in zip(axon[:-1].astype(int), axon[1:].astype(int)):
            transition_counts.loc[condition, f"{start}->{end}"] += 1

reported_metrics = ["Stability", "1->1", "-1->-1", "-1->1", "1->-1", "1->0", "0->1"]
transition_tables = []

for metric in reported_metrics:
    rows = []
    for first, second in combinations(conditions, 2):
        n1, n2 = transition_counts.loc[first].sum(), transition_counts.loc[second].sum()
        if metric == "Stability":
            diagonal = ["-1->-1", "0->0", "1->1"]
            x1 = int(transition_counts.loc[first, diagonal].sum())
            x2 = int(transition_counts.loc[second, diagonal].sum())
        else:
            x1 = int(transition_counts.loc[first, metric])
            x2 = int(transition_counts.loc[second, metric])

        p1, p2, difference, z, pvalue, ci_low, ci_high = compare_two_proportions(x1, n1, x2, n2)
        rows.append({
            "Outcome": metric, "Condition 1": first, "Condition 2": second,
            "x1": x1, "n1": n1, "Proportion 1 (%)": 100 * p1,
            "x2": x2, "n2": n2, "Proportion 2 (%)": 100 * p2,
            "Difference (percentage points)": 100 * difference,
            "95% CI lower": 100 * ci_low, "95% CI upper": 100 * ci_high,
            "z": z, "p": pvalue,
        })

    metric_results = pd.DataFrame(rows)
    metric_results["q (BH-FDR)"] = adjust_pvalues_bh(metric_results["p"])
    metric_results["Significant after FDR"] = metric_results["q (BH-FDR)"] < 0.05
    transition_tables.append(metric_results.sort_values("p"))

transition_pairwise = pd.concat(transition_tables, ignore_index=True)

output_file = OUTPUT_DIR / "phase_transition_results.xlsx"
with pd.ExcelWriter(output_file) as writer:
    phase_summary.to_excel(writer, sheet_name="Phase summary", index=False)
    phase_global.to_excel(writer, sheet_name="Phase global test", index=False)
    phase_pairwise.to_excel(writer, sheet_name="Phase pairwise", index=False)
    transition_counts.to_excel(writer, sheet_name="Transition counts")
    transition_pairwise.to_excel(writer, sheet_name="Transition pairwise", index=False)

display(phase_summary)
display(phase_global)
display(phase_pairwise)
print(f"Results saved to {output_file}")


,Condition,n,mean,sd,median,minimum,maximum
0,CONTROL,64,15.203125,2.966371,15.5,7,20
1,TREATMENT_W,63,12.746032,2.940091,13.0,6,19
2,TREATMENT_X,60,14.700000,3.227530,15.0,6,21
3,TREATMENT_Y,51,17.313725,2.694366,18.0,10,24
4,TREATMENT_Z,47,10.595745,2.894219,11.0,4,16


,Test,Statistic,p
0,Kruskal-Wallis,100.30767,8.459496e-21


,Condition 1,Condition 2,n1,n2,Median 1,Median 2,Mann-Whitney U,p,q (BH-FDR),Significant after FDR
0,TREATMENT_Y,TREATMENT_Z,51,47,18.0,11.0,2292.0,6.522009e-15,6.522009e-14,True
1,TREATMENT_W,TREATMENT_Y,63,51,13.0,18.0,398.5,4.906341e-12,2.453171e-11,True
2,CONTROL,TREATMENT_Z,64,47,15.5,11.0,2602.0,4.995555e-11,1.665185e-10,True
3,TREATMENT_X,TREATMENT_Z,60,47,15.0,11.0,2331.5,6.481621e-09,1.620405e-08,True
4,CONTROL,TREATMENT_W,64,63,15.5,13.0,2938.0,8.063998e-06,1.612800e-05,True
5,TREATMENT_X,TREATMENT_Y,60,51,15.0,18.0,820.0,2.443851e-05,4.073084e-05,True
6,CONTROL,TREATMENT_Y,64,51,15.5,18.0,977.5,2.113510e-04,3.019300e-04,True
7,TREATMENT_W,TREATMENT_Z,63,47,13.0,11.0,2056.5,4.737901e-04,5.922376e-04,True
8,TREATMENT_W,TREATMENT_X,63,60,13.0,15.0,1230.5,8.068968e-04,8.965520e-04,True
9,CONTROL,TREATMENT_X,64,60,15.5,15.0,2126.0,3.013805e-01,3.013805e-01,False


Results saved to /Users/gonzalospelzini/Library/Mobile Documents/com~apple~CloudDocs/Documents/jupyter/results/phase_transition_results.xlsx
